In [ ]:
!pip install datasets sentence-transformers pillow torch torchvision pandas tqdm

In [ ]:
import os
import json
import uuid
from tqdm import tqdm

import torch
import pandas as pd
from PIL import Image

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [ ]:
# Output directories
EMBED_DIR = "embeddings"
META_DIR = "metadata"

os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

CSV_PATH = os.path.join(META_DIR, "embeddings.csv")
JSON_PATH = os.path.join(META_DIR, "embeddings.json")

# Device selection
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
model = SentenceTransformer(
    "sentence-transformers/clip-ViT-B-32",
    device=device
)

In [ ]:
dataset = load_dataset(
    "detection-datasets/coco",
    split="train",
    streaming=True
)

In [ ]:
def crop_object(image, bbox):
    """
    Crop a bounding box from an image.

    bbox format (COCO): [x, y, width, height]
    """
    x, y, w, h = bbox
    return image.crop((x, y, x + w, y + h))

In [ ]:
metadata_rows = []
metadata_json = []

In [ ]:
for sample in tqdm(dataset):
    image: Image.Image = sample["image"]
    image_id = sample["image_id"]
    annotations = sample["annotations"]

    for ann in annotations:
        bbox = ann["bbox"]
        label = ann["category_id"]
        object_id = ann["id"]

        # Crop object
        try:
            crop = crop_object(image, bbox).convert("RGB")
        except Exception:
            continue

        # Generate embedding
        with torch.no_grad():
            embedding = model.encode(
                crop,
                convert_to_tensor=True,
                normalize_embeddings=True
            )

        # Create unique ID and save embedding
        vector_id = str(uuid.uuid4())
        vector_path = os.path.join(EMBED_DIR, f"{vector_id}.pt")

        torch.save(embedding.cpu(), vector_path)

        # Record metadata
        row = {
            "vector_id": vector_id,
            "vector_path": vector_path,
            "image_id": image_id,
            "object_id": object_id,
            "label": label
        }

        metadata_rows.append(row)
        metadata_json.append(row)

In [ ]:
df = pd.DataFrame(metadata_rows)
df.to_csv(CSV_PATH, index=False)

print(f"Saved CSV metadata to {CSV_PATH}")

In [ ]:
with open(JSON_PATH, "w") as f:
    json.dump(metadata_json, f, indent=2)

print(f"Saved JSON metadata to {JSON_PATH}")